In [1]:
import sys
sys.path.append("../")
import os
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import requests
from src.chunking import chunk_text
from src.loaders.txt_loader import load_txt

d:\Program\anaconda3\envs\rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["HF_HOME"] = r"G:\huggingface_cache"

In [13]:
# 1. embedding模型（轻量）
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5317.02it/s]


In [3]:
def load_folder(folder):
    docs = []
    for file in os.listdir(folder):
        if file.endswith(".txt"):
            text = load_txt(f"{folder}/{file}")
            docs.append({
                "source": file,
                "text": text
            })
    return docs

In [5]:
loaded_docs = load_folder("../data")
print(loaded_docs)

[{'source': 'coffee.txt', 'text': 'Coffee is one of the most widely consumed beverages in the world. Although many people associate it mainly with energy and productivity, coffee culture varies significantly between regions.\n\nCoffee begins as a fruit grown in tropical environments. After harvesting, beans go through processing stages that influence sweetness, acidity, and body. Roasting introduces another layer of variation. Light roasts tend to preserve more original characteristics, while darker roasts emphasize bitterness and roasted flavors.\n\nBrewing methods also change the final result. Pour-over techniques are often valued for clarity and control. Espresso uses pressure to produce concentrated flavor and serves as the base for drinks such as cappuccino and latte. Cold brew extracts flavor differently and is commonly perceived as smoother.\n\nIn recent years, specialty coffee has encouraged consumers to think more about origin and preparation methods. Terms such as single-orig

In [ ]:
docs = []
chunk_id = 0
for file in loaded_docs:
    chunks = chunk_text(
        file["text"],
        chunk_size=200,
        overlap=50
    )
    for chunk in chunks:
        docs.append({
            "chunk_id": chunk_id,
            "source": file["source"],
            "text": chunk
        })
        chunk_id += 1

In [12]:
docs[:10]

[{'chunk_id': 0,
  'source': 'coffee.txt',
  'text': 'Coffee is one of the most widely consumed beverages in the world. Although many people associate it mainly with energy and productivity, coffee culture varies significantly between regions.\n\nCoffee be'},
 {'chunk_id': 1,
  'source': 'coffee.txt',
  'text': 'e varies significantly between regions.\n\nCoffee begins as a fruit grown in tropical environments. After harvesting, beans go through processing stages that influence sweetness, acidity, and body. Roas'},
 {'chunk_id': 2,
  'source': 'coffee.txt',
  'text': ' that influence sweetness, acidity, and body. Roasting introduces another layer of variation. Light roasts tend to preserve more original characteristics, while darker roasts emphasize bitterness and '},
 {'chunk_id': 3,
  'source': 'coffee.txt',
  'text': 'ics, while darker roasts emphasize bitterness and roasted flavors.\n\nBrewing methods also change the final result. Pour-over techniques are often valued for clarity an

In [14]:
# 3. 向量化
embeddings = model.encode([d["text"] for d in docs])
embeddings = np.array(embeddings).astype("float32")

In [15]:
# 4. 建立FAISS索引
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

In [16]:
def retrieve(query, k=2):
    q_emb = model.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    distances, indices = index.search(q_emb, k)
    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            "chunk_id": docs[idx]["chunk_id"],
            "source": docs[idx]["source"],
            "score": float(distances[0][rank]),
            "text": docs[idx]["text"],
            "cosine_similarity": float(distances[0][rank])
        })
    return results

def ask_llm(context, question):
    prompt = f"""
You are a helpful assistant.

Context:
{context}

Question:
{question}
"""

    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen2.5:7b",
            "prompt": prompt,
            "stream": False
        }
    )

    return res.json()["response"]

In [18]:
if __name__ == "__main__":
    query = "What is cosine similarity?"

    retrieved = retrieve(query)
    context = "\n".join(r["text"] for r in retrieved)

    answer = ask_llm(context, query)

    print("\n=== Retrieved ===")

    for r in retrieved:
        print(
            f"""
            Chunk {r['chunk_id']}
            score {r["source"]}
            cosine_similarity:{r['cosine_similarity']:.3f}
            {r['text']}
            """
        )

    print("\n=== Answer ===")
    print(answer)


=== Retrieved ===

            Chunk 33
            score sample.txt
            cosine_similarity:0.510
            .
Vector normalization converts vectors to unit length.
After normalization, inner product becomes equivalent to cosine similarity.
FAISS supports both L2 distance and inner product search.
Top-K retr
            

            Chunk 32
            score sample.txt
            cosine_similarity:0.487
             sentence embeddings.
Semantic similarity can be measured using cosine similarity.
Cosine similarity focuses on vector direction rather than magnitude.
Vector normalization converts vectors to unit le
            

=== Answer ===
Cosine similarity is a measure of similarity between two non-zero vectors in a multi-dimensional space. It measures the cosine of the angle between the two vectors, which provides a useful metric for determining how similar the vectors are irrespective of their magnitude.

In more detail:

- **Formula**: The cosine similarity between two